# 02 · Memory — 01 short-term memory (truncation inside one conversation)

**Short-term memory is what the agent carries from turn to turn inside a single conversation — and it is finite, because the context window is finite.** Nothing here is persisted; close the kernel and it is gone. `02-long-term.ipynb` is the other half.

Every turn of a chat agent re-sends the whole conversation. The model has no memory of the last call — the "memory" is entirely a string the caller rebuilds each time. That string has a hard ceiling, so something has to be dropped, and *which* thing gets dropped is a design decision that a product usually makes once, early, in four lines of code, and then lives with.

This notebook ports those four lines from a real product: SafeBite's `_truncate_history` (`src/safe_bite/pipeline/orchestrator.py`). It keeps the last 4 messages and caps assistant messages at 150 characters. It is ported faithfully, including its flaws — which are demonstrated rather than fixed, because the flaws are the lesson.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `estimate_tokens` | Rough token count of a string or a message list, so "the window is finite" is a number and not a feeling. | `estimate_tokens(history)` → `312` |
| `truncate_history` | The ported SafeBite policy: keep the last 4 messages, cap assistant content at 150 chars. | `truncate_history(history)` → 4 messages |
| `build_prompt` | Assembles context + recent history + the new query into the single string the model actually sees. | `build_prompt(ctx, hist, query)` |
| `stand_in_agent` | A deterministic, no-key stand-in that answers *only* from what is in the prompt, so "the fact fell out of the window" becomes a visible wrong answer. | `stand_in_agent(prompt, "kung pao chicken")` → `"YES"` / `"NO"` |
| `truncate_to_budget` | What a token-budgeted policy with pinned facts looks like instead — shown as a contrast, not as a patch to the port. | `truncate_to_budget(history, 200, pinned)` |

## Step 1 — bootstrap the repo path and confirm the environment

Jupyter starts this kernel with the notebook's own directory as `cwd`, so `nbio` has to be located and put on `sys.path` before anything can import it. `show_environment()` then prints which keys are loaded, so a reader knows whether the optional real-model step near the end will run or skip.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

# bootstrap() above already put the repo root on sys.path and loaded .env;
# `_root` is that same path, reused here rather than resolved a second time.
repo_root = _root
env = nbio.show_environment()

## Step 2 — a conversation, as the caller actually stores it

A list of `{"role", "content"}` dicts. This is the entire short-term memory of a chat agent: no database, no object, just a list the caller owns and re-sends on every turn.

The fixture is synthetic and written for this notebook — no real user, no real menu. Turn 1 carries the fact that matters: a peanut allergy.

In [ ]:
history = [
    {"role": "user", "content": "Hi — before we start, I should say I'm allergic to peanuts. Badly. Please keep that in mind for anything you suggest."},
    {"role": "assistant", "content": "Noted. I'll flag anything that contains that ingredient before recommending it, and I'll check the ingredient list rather than guessing from the dish name."},
    {"role": "user", "content": "What's popular on the lunch menu today?"},
    {"role": "assistant", "content": "Today's most-ordered lunch dishes are the Kung Pao Chicken, the Paneer Tikka Roll, the Chana Masala thali, and the Lemon Rice bowl. The thali is the best value of the four."},
    {"role": "user", "content": "Do you do anything vegetarian?"},
    {"role": "assistant", "content": "Yes — the Paneer Tikka Roll, the Chana Masala thali, and the Lemon Rice bowl are all vegetarian. The thali is the only one of the three that is also vegan."},
    {"role": "user", "content": "What comes with the Kung Pao Chicken?"},
    {"role": "assistant", "content": "It's served with steamed jasmine rice and a side of stir-fried bok choy. You can swap the rice for brown rice at no extra charge."},
]

query = "Is the Kung Pao Chicken a good pick for me?"

for i, m in enumerate(history):
    print(f"  [{i}] {m['role']:<9} {m['content'][:72]}...")
print()
print(f"  new query: {query!r}")

## Step 3 — why a finite window forces the question at all

`estimate_tokens` is the crude 4-characters-per-token rule of thumb. It is not a tokenizer and does not pretend to be one — it is here so the ceiling is a number on screen rather than an abstraction.

The point of this step: the conversation grows monotonically, and the budget does not. Every turn added is a turn closer to the wall.

In [ ]:
def estimate_tokens(obj) -> int:
    """Rough token count: ~4 characters per token. A real system would call
    the model's own tokenizer; this is the back-of-envelope version, and it
    is only ever used here to compare sizes against a budget."""
    if isinstance(obj, str):
        return max(1, len(obj) // 4)
    if isinstance(obj, dict):
        return estimate_tokens(obj.get("content", ""))
    return sum(estimate_tokens(m) for m in obj)


HISTORY_BUDGET_TOKENS = 120  # the slice of the window this agent gives to history

rows = []
for n in range(1, len(history) + 1):
    used = estimate_tokens(history[:n])
    rows.append((n, used, HISTORY_BUDGET_TOKENS, "OVER BUDGET" if used > HISTORY_BUDGET_TOKENS else "fits"))

nbio.table(rows, ("messages", "est. tokens", "budget", "status"))
print()
print(f"full conversation: {estimate_tokens(history)} est. tokens against a {HISTORY_BUDGET_TOKENS}-token budget")
assert estimate_tokens(history) > HISTORY_BUDGET_TOKENS, "the fixture must actually exceed the budget, or there is nothing to demonstrate"

## Step 4 — the ported policy: `truncate_history`

This is SafeBite's `_truncate_history`, ported as written. Two rules, both of them hardcoded constants:

1. keep the **last 4 messages** (2 turns);
2. if a message is from the assistant and longer than **150 characters**, cut it at 150 and append `"..."`.

User messages are never shortened. Nothing is summarised. Nothing that falls off the front is recorded anywhere.

In [ ]:
def truncate_history(chat_history):
    """Truncate chat history to save tokens.

    Keeps the last 4 messages (2 turns) and caps assistant messages at
    150 characters. Ported from SafeBite's orchestrator.
    """
    if not chat_history:
        return []

    recent = chat_history[-4:]  # last 4 messages (2 turns)
    truncated = []
    for msg in recent:
        role = msg.get("role", "user")
        content = msg.get("content", "")
        if role == "assistant" and len(content) > 150:
            content = content[:150] + "..."
        truncated.append({"role": role, "content": content})
    return truncated

## Step 5 — run it, and look at exactly what survived

Real output, both sides. The `delta` line reports collection-level counts; the listing under it is what a reader actually needs to see — which messages are gone.

In [ ]:
kept = truncate_history(history)

nbio.delta(history, kept, label="truncate_history")
print()

print("SURVIVED:")
for m in kept:
    mark = " (cut at 150)" if m["content"].endswith("...") else ""
    print(f"  {m['role']:<9} {m['content'][:60]}...{mark}")
print()
print("DROPPED ENTIRELY:")
for m in history[: len(history) - 4]:
    print(f"  {m['role']:<9} {m['content'][:60]}...")

assert len(kept) == 4
assert estimate_tokens(kept) < estimate_tokens(history)
print()
print(f"est. tokens: {estimate_tokens(history)} -> {estimate_tokens(kept)}")

## Step 6 — build the prompt the model actually sees

`build_prompt` is the same shape SafeBite assembles: a context line, a `Recent:` block of the truncated history, then the new user turn. This string *is* the agent's short-term memory. Anything not in it does not exist as far as the model is concerned.

In [ ]:
def build_prompt(context_parts, truncated_history, user_query) -> str:
    """Assemble the single string sent to the model: context, recent turns, query."""
    history_text = ""
    if truncated_history:
        lines = []
        for msg in truncated_history:
            prefix = "User" if msg["role"] == "user" else "Assistant"
            lines.append(f"{prefix}: {msg['content']}")
        history_text = "\nRecent:\n" + "\n".join(lines) + "\n"
    return f"""Context: {' | '.join(context_parts)}
{history_text}
User: {user_query}"""


context = ["Tenant: demo-bistro", "Preferred language: English"]

prompt_full = build_prompt(context, history, query)
prompt_truncated = build_prompt(context, kept, query)

print(prompt_truncated)
print()
print(f"full-history prompt : {estimate_tokens(prompt_full)} est. tokens")
print(f"truncated prompt    : {estimate_tokens(prompt_truncated)} est. tokens")

## Step 7 — the fact is provably gone

Not "probably dropped" — checked. The allergy string appears in the full prompt and does not appear in the truncated one.

In [ ]:
assert "peanut" in prompt_full.lower(), "the fixture must state the allergy in the full history"
assert "peanut" not in prompt_truncated.lower(), "the truncated prompt must have lost it — that is the whole demonstration"

print("'peanut' in full prompt      :", "peanut" in prompt_full.lower())
print("'peanut' in truncated prompt :", "peanut" in prompt_truncated.lower())

## Step 8 — what that costs: a deterministic stand-in agent

`stand_in_agent` answers strictly from the text of the prompt it is handed. It is not a model and makes no claim to be one — it is a way to make the consequence of a dropped fact *visible and deterministic*, with no key and no network. A real model given the same two prompts behaves the same way for the same reason: it cannot use a fact it was never sent.

The dish's ingredient list comes from the menu, not from the chat — so the agent knows Kung Pao Chicken contains peanuts. What it does not know, after truncation, is that this particular user cannot eat them.

In [ ]:
KNOWN_ALLERGENS = ("peanut", "shellfish", "gluten", "dairy", "soy")

MENU_INGREDIENTS = {
    "kung pao chicken": {"chicken", "peanut", "chili", "soy"},
    "paneer tikka roll": {"paneer", "dairy", "gluten"},
    "chana masala thali": {"chickpea", "tomato", "gluten"},
    "lemon rice bowl": {"rice", "lemon", "peanut"},
}


def stand_in_agent(prompt: str, dish: str) -> str:
    """Answer using only what is present in the prompt string.

    Reads the user's stated allergens out of the prompt text, intersects them
    with the dish's ingredients, and refuses on any overlap.
    """
    text = prompt.lower()
    stated = {a for a in KNOWN_ALLERGENS if a in text}
    ingredients = MENU_INGREDIENTS[dish]
    conflict = sorted(stated & ingredients)
    if conflict:
        return f"NO — {dish} contains {', '.join(conflict)}, which you told me you react to."
    return f"YES — {dish} looks like a good pick."


answer_full = stand_in_agent(prompt_full, "kung pao chicken")
answer_truncated = stand_in_agent(prompt_truncated, "kung pao chicken")

print("with full history :", answer_full)
print("after truncation  :", answer_truncated)

assert answer_full.startswith("NO"), "with the allergy in the window the agent must refuse"
assert answer_truncated.startswith("YES"), "without it the agent recommends the dish — that is the cost"

## Step 9 — the same prompts through a real model, if a key is set

The stand-in above is deterministic by construction, which is exactly what makes it a weak witness: someone could reasonably suspect the outcome was rigged by the matcher. This step sends the two prompts to a real model instead. With no key it prints that plainly and skips — the notebook's claim already stands on Step 7, which is a string check on the prompt and needs no model at all.

The call is wrapped in `nbio.cost_meter` with a 50-cent ceiling, which is far more than two short chat completions cost.

In [ ]:
import os

GROQ_MODEL = "llama-3.1-8b-instant"
has_groq = bool(env.get("GROQ_API_KEY"))

SYSTEM = (
    "You are a restaurant assistant. Answer in one sentence: is this dish safe "
    "and a good pick for this specific user? Use only what the conversation tells you."
)
MENU_LINE = "\nMenu fact: Kung Pao Chicken contains chicken, peanuts, chili and soy.\n"

with nbio.cost_meter(budget_usd=0.50) as meter:
    if not has_groq:
        print(
            "GROQ_API_KEY not set, skipping — deterministic stand-in instead.\n"
            "Step 8 above already shows the behaviour, and Step 7 proves the cause "
            "(the fact is absent from the prompt) without any model at all."
        )
    else:
        from groq import Groq

        client = Groq(api_key=os.environ["GROQ_API_KEY"])
        for label, p in (("full history", prompt_full), ("truncated", prompt_truncated)):
            r = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM},
                    {"role": "user", "content": p + MENU_LINE},
                ],
                max_tokens=80,
                temperature=0,
            )
            meter.record(GROQ_MODEL, r.usage.prompt_tokens, r.usage.completion_tokens)
            print(f"{label:<13}: {r.choices[0].message.content.strip()}")

print()
print(meter.report())

## Step 10 — flaw one, stated honestly: the cut is by message count, not by tokens

The ported policy keeps 4 messages regardless of how large they are, and it never shortens a user message. So a user who writes four long paragraphs blows the budget *after* truncation, and the function reports success. The thing it was written to guarantee is the one thing it does not guarantee.

This is the port's behaviour, not a strawman — it is what `chat_history[-4:]` plus an assistant-only length cap does.

In [ ]:
long_history = [
    {"role": "user", "content": "I am planning a large office order and here are the constraints. " * 12},
    {"role": "assistant", "content": "Understood, let me check availability for a group of that size."},
    {"role": "user", "content": "Also several people have restrictions, listed at length as follows. " * 12},
    {"role": "assistant", "content": "Got it — I will cross-check each restriction against the menu."},
]

after = truncate_history(long_history)

print(f"messages in : {len(long_history)}   est. tokens: {estimate_tokens(long_history)}")
print(f"messages out: {len(after)}   est. tokens: {estimate_tokens(after)}")
print(f"budget      : {HISTORY_BUDGET_TOKENS}")
print()

assert len(after) == len(long_history), "nothing was dropped — all four messages are inside the keep-4 window"
assert estimate_tokens(after) > HISTORY_BUDGET_TOKENS, "and the result is still over budget"
print("truncation ran, dropped nothing, and left the prompt over budget by "
      f"{estimate_tokens(after) - HISTORY_BUDGET_TOKENS} est. tokens.")

## Step 11 — flaw two: the 150-character cap cuts mid-sentence

The cap is a raw character slice. If the operative clause of an assistant message sits past character 150, it is severed — and the `"..."` gives no hint that what was lost was the important half rather than pleasantries.

In [ ]:
warning = {
    "role": "assistant",
    "content": (
        "Thanks for waiting. I checked with the kitchen about today's lunch service "
        "and there are a couple of small changes to the usual preparation worth "
        "mentioning: the sauce base was substituted this morning and now contains peanut oil."
    ),
}

cut = truncate_history([warning])[0]["content"]

print("ORIGINAL:")
print(" ", warning["content"])
print()
print("AFTER THE 150-CHAR CAP:")
print(" ", cut)
print()

assert "peanut oil" in warning["content"]
assert "peanut oil" not in cut, "the operative clause sat past character 150 and was severed"
print("The warning survived as prose and did not survive as information.")

## Step 12 — what a budget-aware policy looks like instead

Not a patch to the port — a contrast, so the difference is concrete. Two changes: fill to a **token** budget rather than a message count, and **pin** facts that must never age out. Pinned facts are extracted once and carried as context, so they cost a fixed handful of tokens forever instead of competing with recency.

Pinning is the honest name for what is happening: someone decided in advance that allergies matter more than what came with the rice. Nothing here discovers that on its own.

In [ ]:
def extract_pinned_facts(chat_history) -> list[str]:
    """Scan for statements that must outlive the window. Keyword-matched on
    purpose — a real system would use a classifier or an explicit user
    profile, and this is neither."""
    facts = []
    for m in chat_history:
        if m["role"] != "user":
            continue
        low = m["content"].lower()
        if "allerg" in low:
            for a in KNOWN_ALLERGENS:
                if a in low and f"allergic to {a}" not in facts:
                    facts.append(f"allergic to {a}")
    return facts


def truncate_to_budget(chat_history, budget_tokens: int, pinned: list[str]):
    """Keep pinned facts plus as many of the most recent messages as fit."""
    spent = estimate_tokens(" | ".join(pinned)) if pinned else 0
    out = []
    for msg in reversed(chat_history):
        cost = estimate_tokens(msg)
        if spent + cost > budget_tokens:
            break
        out.append(msg)
        spent += cost
    return list(reversed(out)), spent


pinned = extract_pinned_facts(history)
budgeted, spent = truncate_to_budget(history, HISTORY_BUDGET_TOKENS, pinned)

prompt_budgeted = build_prompt(context + [f"Known about this user: {', '.join(pinned)}"], budgeted, query)
answer_budgeted = stand_in_agent(prompt_budgeted, "kung pao chicken")

print(f"pinned facts   : {pinned}")
print(f"messages kept  : {len(budgeted)} of {len(history)}")
print(f"est. tokens    : {spent} against a budget of {HISTORY_BUDGET_TOKENS}")
print(f"answer         : {answer_budgeted}")
print()

assert spent <= HISTORY_BUDGET_TOKENS, "the budgeted policy must actually stay under budget"
assert answer_budgeted.startswith("NO"), "and the pinned fact must survive to change the answer"

# The same two policies on Step 10's oversized conversation, where the
# difference is the budget rather than the pinned fact.
long_kept = truncate_history(long_history)
long_budgeted, long_spent = truncate_to_budget(long_history, HISTORY_BUDGET_TOKENS, [])

nbio.table(
    [
        ("this conversation", "truncate_history (ported)", len(kept), estimate_tokens(kept),
         answer_truncated.split(" — ")[0]),
        ("this conversation", "truncate_to_budget", len(budgeted), spent,
         answer_budgeted.split(" — ")[0]),
        ("Step 10's long one", "truncate_history (ported)", len(long_kept),
         estimate_tokens(long_kept), "n/a"),
        ("Step 10's long one", "truncate_to_budget", len(long_budgeted), long_spent, "n/a"),
    ],
    ("conversation", "policy", "msgs kept", "est. tokens", "answer"),
)

assert long_spent <= HISTORY_BUDGET_TOKENS, "the budgeted policy holds where the ported one did not"
print()
print("On this conversation both keep 4 messages and the pinned fact is what differs.")
print("On Step 10's, the ported policy keeps all 4 and blows the budget; the budgeted")
print(f"one keeps {len(long_budgeted)} and stays under it.")
if not long_budgeted:
    print()
    print("Worth naming: it keeps zero, because no single message in that conversation")
    print("fits the budget on its own. A budget alone does not remove the need to")
    print("shorten an individual message — it only makes the failure visible instead")
    print("of silent.")

## Step 13 — and the limit of pinning

Short-term memory that pins a fact still forgets it the moment the conversation ends. A new session starts with an empty list, and the user has to declare the allergy again. Nothing in this notebook writes anything to disk.

In [ ]:
new_session_history = []
prompt_new_session = build_prompt(context, truncate_history(new_session_history), query)
answer_new_session = stand_in_agent(prompt_new_session, "kung pao chicken")

print("new session, empty history:")
print(" ", answer_new_session)

assert truncate_history(new_session_history) == []
assert answer_new_session.startswith("YES"), "a fresh conversation knows nothing about this user"
print()
print("Carrying a decision past the end of a conversation is a different mechanism — 02-long-term.ipynb.")

## Where this fits

`01-tools/` is how an agent *does* things. This stage is what it *carries*. Short-term memory is the cheapest kind — a list in the caller's process — and the first one every agent hits the limit of.

Next: `02-long-term.ipynb` writes a decision to disk and recovers it in a different process. `03-semantic-recall.ipynb` then asks the harder question — how to find the right past turn when there are thousands of them and recency is the wrong sort order.

## What did not come across

- **SafeBite's follow-up detection and cached-context injection.** `_is_followup_message` and `_build_followup_context` re-inject a cached profile and menu summary so the model can skip a tool call. That is a real and effective mitigation for exactly the loss demonstrated here, but it depends on a tenant-scoped menu cache and a live profile store, neither of which travels into this repo.
- **The agent's system prompt and the `recommendation_agent` object.** The port here is `_truncate_history` and the prompt assembly around it; the framework agent it feeds, the tool set, and the lock that serialises calls to it are product code.
- **A real tokenizer.** `estimate_tokens` is 4-chars-per-token. Every token figure printed above is an estimate and will differ from what a provider bills. The *shape* of the argument — history grows, the budget does not — does not depend on the constant.
- **Multi-user and multi-tenant isolation.** One conversation, one user, one process. Nothing here addresses two users' histories sharing a cache.